# Tutorial 2 — A household as a two-equation system

**Source:** `Tutorial_2.py`

**In class:** PS1 Question 3(f); about 30 minutes of programming.

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. Last week's model with an adaptive solver

`solve_ivp` expects a function of time and a **vector** of state variables. Here the vector has one element, $k$. Compare its answer with a short Euler loop.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


In [ ]:
ALPHA, A, S, DELTA, N = 1 / 3, 1.0, 0.20, 0.05, 0.01
k_star = (S * A / (DELTA + N)) ** (1 / (1 - ALPHA))


def solow(t, y):
    k = y[0]
    return [S * A * k**ALPHA - (DELTA + N) * k]


In [ ]:
sol = solve_ivp(solow, (0, 50), [k_star/4], dense_output=True)
print('Adaptive solver k(50):', sol.sol(50)[0])


In [ ]:
k, dt = k_star/4, 0.01
for _ in range(int(50/dt)):
    k += dt*(S*A*k**ALPHA-(DELTA+N)*k)
print('Euler loop k(50):', k)


## 2. Wealth and consumption move together

For the household,

$$\dot a=ra+w-c,\qquad \dot c=gc,\qquad g=\frac{r-\rho}{\sigma}.$$

The state vector is $(a,c)$. Keep this order consistent when passing initial values and reading solver output.


In [ ]:
R, RHO, SIGMA, W, A0 = 0.05, 0.03, 2.0, 1.0, 1.0
G = (R - RHO) / SIGMA


def household(t, y):
    a, c = y
    return [R * a + W - c, G * c]


In [ ]:
c0_star = (R-G)*(A0+W/R)
print(f'g = {G:.3f}; optimal initial consumption = {c0_star:.4f}')


## 3. What does transversality test?

Part (d) gives $c_0^*$. Assets themselves need not tend to zero. The condition concerns **discounted** assets, $e^{-rt}a(t)$. Predict which column below shrinks.


In [ ]:
opt = solve_ivp(household, (0, 200), [A0, c0_star],
                dense_output=True, rtol=1e-10, atol=1e-12)
for year in (0, 50, 100, 200):
    a = opt.sol(year)[0]
    print(f't={year:3d}: a={a:10.3f}; discounted a={np.exp(-R*year)*a:.5f}')


## 4. A terminal event catches wealth exhaustion

If initial consumption is too high, assets cross zero. An event returns the quantity whose zero matters. `direction=-1` asks for a downward crossing.


In [ ]:
def assets_exhausted(t, y):
    return y[0]


assets_exhausted.terminal = True
assets_exhausted.direction = -1  # only on the way down, not on the way up


In [ ]:
high = solve_ivp(household, (0, 200), [A0, 1.01*c0_star],
                 events=assets_exhausted, rtol=1e-10, atol=1e-12)
print('Stopped at t =', high.t_events[0][0])
print('Final assets =', high.y[0, -1])


## 5. Compare guesses on both sides

A guess below the optimum has no zero crossing; it grows away instead. What would a numerical stopping rule need to detect that case?


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, c0 in [('1% low', .99*c0_star), ('optimal', c0_star), ('1% high', 1.01*c0_star)]:
    s = solve_ivp(household, (0, 200), [A0, c0], events=assets_exhausted,
                  rtol=1e-10, atol=1e-12)
    ax.plot(s.t, s.y[0], label=label)
    print(label, 'event time:', s.t_events[0][0] if len(s.t_events[0]) else 'none')
ax.set(xlabel='time', ylabel='assets $a$', xlim=(0, 200), ylim=(-20, 200))
ax.legend(frameon=False); plt.show()


**Pause and connect to PS1 3(f).** Explain why an event catches overconsumption cleanly but underconsumption needs a chosen threshold or horizon. Next week we get two detectable failures and can bisect between them.
